Es posible que los modelos se estanquen en una Barren Plateau porque el data qubit esté demasiado entrelazado con los ancilla qubit, que aportan ruido (sobretodo el Haar state). Vamos a comparar cuánto se parece el output de ese data qubit a un maximally mixed state (I/2).

In [18]:
from src.utils import get_path_MSQDDPM, set_device
from src.MSQDDPM_angel import MSQDDPM
import torch
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from tqdm import tqdm
config = {
    'device': 'cpu',
    'seed': 0,
    'dataset': {
        'dir': './results_MSQDDPM',
        'name': 'CIRCLEY_1',
        # 'transforms': {
        #     'resize': 16
        # },
        'maxsize': 300,
    },
    'model': {
        'n_timesteps': 6,
        'n_zero_ancilla_qubits': 1,
        'n_haar_ancilla_qubits': 1,
        'n_backward_layers': 8,
        'diffusion_schedule': {
            'name': 'linear',
            # 'slope': 2,
            # 'vrescale': 50,
            # 'hrescale': 26
        }
    }
}
n_qubits = 1
n_features = 2**n_qubits
diffusion_schedule_nickname = config['model']['diffusion_schedule']['name']
n_zero_ancilla_qubits = config['model']['n_zero_ancilla_qubits']
n_haar_ancilla_qubits = config['model']['n_haar_ancilla_qubits']
n_backward_layers = config['model']['n_backward_layers']
seed = config['seed']
n_data = config['dataset']['maxsize']
n_timesteps = config['model']['n_timesteps']
device = set_device(config.get('device', 'cpu'))
torch.set_default_device(device)
get_path = partial(get_path_MSQDDPM, config, n_data=n_data, n_features=n_features, n_qubits=n_qubits, n_zero_ancilla_qubits=n_zero_ancilla_qubits, n_haar_ancilla_qubits=n_haar_ancilla_qubits, n_timesteps=n_timesteps, n_backward_layers=n_backward_layers)
None

Using device: cpu


In [40]:
dir, filename = get_path(type='diffusedqstates.npy', diffusion_schedule=diffusion_schedule_nickname)
dm_diffused = np.load(dir/filename)
print(f"Diffused states shape: {dm_diffused.shape}")
model = MSQDDPM(n_qubits, n_zero_ancilla_qubits, n_haar_ancilla_qubits, n_timesteps, n_backward_layers, seed=seed).to(device)

# Inference until TIMESTEP=5
TIMESTEP = 3

# Load params
params_tot = torch.zeros((n_timesteps, 2*(n_qubits+n_zero_ancilla_qubits+n_haar_ancilla_qubits)*n_backward_layers), device=device)
for tt in range(TIMESTEP, n_timesteps):
    dir, filename = get_path(type='bestparams.npy', t=tt)
    params_tot[tt] = torch.from_numpy(np.load(dir / filename)).to(device)

inputs_last_timestep = torch.from_numpy(dm_diffused[-1]).to(device)
input_tplus1 = model.prepareInput_t(inputs_last_timestep, params_tot, TIMESTEP, n_data)
measured_full = model.backwardOutput_t(input_tplus1, params_tot[TIMESTEP-1])
output = model._trace_out_ancilla_vmap(measured_full)
print(f"Output shape: {output.shape}")
print(output[0])
        

Diffused states shape: (7, 300, 2, 2)
Output shape: torch.Size([300, 2, 2])
tensor([[ 0.5660+8.2503e-09j, -0.1298+1.6585e-03j],
        [-0.1298-1.6586e-03j,  0.4340-8.2503e-09j]])


In [41]:
# How close is the output to the maximal mixed state?
maximally_mixed = torch.eye(2**n_qubits, dtype=torch.complex64) / 2**n_qubits

torch.allclose(output, maximally_mixed, atol=0.1)

False